In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install tqdm scikit-learn

import os, time, copy
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import classification_report, confusion_matrix


In [ ]:
DATA_ROOT = "/content/drive/MyDrive/MERGED_COLOR_128/"

TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR   = os.path.join(DATA_ROOT, "val")
TEST_DIR  = os.path.join(DATA_ROOT, "test")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
print(TRAIN_DIR)
print(VAL_DIR)
print(TEST_DIR)

device: cuda
/content/drive/MyDrive/MERGED_COLOR_128/train
/content/drive/MyDrive/MERGED_COLOR_128/val
/content/drive/MyDrive/MERGED_COLOR_128/test


In [ ]:
import torchvision.transforms.functional as F
from PIL import Image
from torchvision import transforms

class PadToSquare:
    def __init__(self, fill=0):
        self.fill = fill

    def __call__(self, img: Image.Image):
        w, h = img.size
        if w == h:
            return img
        size = max(w, h)
        pad_left = (size - w) // 2
        pad_top = (size - h) // 2
        pad_right = size - w - pad_left
        pad_bottom = size - h - pad_top
        return F.pad(img, (pad_left, pad_top, pad_right, pad_bottom), fill=self.fill)

IMG_SIZE = 128

train_tfms = transforms.Compose([
    PadToSquare(fill=0),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

eval_tfms = transforms.Compose([
    PadToSquare(fill=0),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])


In [ ]:
train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_tfms)
val_ds   = datasets.ImageFolder(VAL_DIR, transform=eval_tfms)
test_ds  = datasets.ImageFolder(TEST_DIR, transform=eval_tfms)

class_names = train_ds.classes
num_classes = len(class_names)
print("classes:", class_names)
print("num_classes:", num_classes)

BATCH_SIZE = 32        # turunin dulu
NUM_WORKERS = 0        # paling penting buat debugging
PIN_MEMORY = False     # matiin dulu biar simpel

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)


classes: ['beige', 'black', 'blue', 'brown', 'gray', 'green', 'pink', 'purple', 'red', 'white', 'yellow']
num_classes: 11


In [ ]:
from collections import Counter

targets = [y for _, y in train_ds.samples]
count = Counter(targets)
freq = np.array([count[i] for i in range(num_classes)], dtype=np.float32)

# weight = total/(num_classes*freq)  (lebih stabil daripada 1/freq doang)
weights = (freq.sum() / (num_classes * freq))
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

print("class freq:", {class_names[i]: int(freq[i]) for i in range(num_classes)})
print("class weights:", {class_names[i]: float(weights[i]) for i in range(num_classes)})


class freq: {'beige': 821, 'black': 9613, 'blue': 3985, 'brown': 1447, 'gray': 9879, 'green': 2331, 'pink': 483, 'purple': 536, 'red': 4995, 'white': 6409, 'yellow': 4515}
class weights: {'beige': 4.984386920928955, 'black': 0.42569246888160706, 'blue': 1.026896357536316, 'brown': 2.82804536819458, 'gray': 0.4142303764820099, 'green': 1.7555477619171143, 'pink': 8.472426414489746, 'purple': 7.63466739654541, 'red': 0.8192555904388428, 'white': 0.6385055184364319, 'yellow': 0.9063525795936584}


In [ ]:
model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)

# Ganti head classifier terakhir
in_features = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(in_features, num_classes)

model = model.to(device)

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 132MB/s]


In [ ]:
criterion = torch.nn.CrossEntropyLoss().cuda()
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)



In [ ]:
import torch

assert torch.cuda.is_available(), "❌ CUDA tidak tersedia! Training dipaksa pakai GPU."

device = torch.device("cuda")
print("✅ Using device:", torch.cuda.get_device_name(0))


✅ Using device: NVIDIA L4


In [ ]:
model = model.to(device)

In [ ]:
from tqdm import tqdm
import time

def run_epoch(model, loader, criterion, optimizer=None, train=False):
    model.train() if train else model.eval()

    running_loss, correct, total = 0.0, 0, 0
    t_batch0 = time.time()

    for i, (x, y) in enumerate(tqdm(loader, leave=False)):
        x = x.cuda(non_blocking=True)
        y = y.cuda(non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            outputs = model(x)
            loss = criterion(outputs, y)
            if train:
                loss.backward()
                optimizer.step()

        running_loss += loss.item() * x.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

        # debug: cek batch pertama lama nggak
        if i == 0:
            print(f"first batch time: {time.time()-t_batch0:.2f}s | x {tuple(x.shape)}")

    return running_loss / total, correct / total


In [ ]:
EPOCHS = 50
best_loss = float("inf")
best_wts = copy.deepcopy(model.state_dict())

PATIENCE = 5
MIN_DELTA = 1e-4

patience_counter = 0

for epoch in range(1, EPOCHS+1):
    t0 = time.time()

    train_loss, train_acc = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer=optimizer,
        train=True
    )

    val_loss, val_acc = run_epoch(
        model,
        val_loader,
        criterion,
        train=False
    )

    scheduler.step(val_loss)

    improved = val_loss < best_loss - MIN_DELTA

    if improved:
        best_loss = val_loss
        best_wts = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss {train_loss:.4f} acc {train_acc:.4f} | "
        f"val loss {val_loss:.4f} acc {val_acc:.4f} | "
        f"best_loss {best_loss:.4f} | "
        f"patience {patience_counter}/{PATIENCE} | "
        f"time {time.time()-t0:.1f}s"
    )

    if patience_counter >= PATIENCE:
        print(f"Early stopping triggered at epoch {epoch}")
        break

model.load_state_dict(best_wts)
print("Training done. Best val loss:", best_loss)


  0%|          | 1/1407 [00:14<5:28:23, 14.01s/it]

first batch time: 14.01s | x (32, 3, 128, 128)


  2%|▏         | 1/49 [00:12<10:03, 12.57s/it]

first batch time: 12.57s | x (32, 3, 128, 128)


Epoch 01/50 | train loss 0.3561 acc 0.8796 | val loss 0.4078 acc 0.8748 | best_loss 0.4078 | patience 0/5 | time 18552.6s


...

  0%|          | 1/1407 [00:00<03:42,  6.33it/s]

first batch time: 0.16s | x (32, 3, 128, 128)


  4%|▍         | 2/49 [00:00<00:03, 12.15it/s]

first batch time: 0.09s | x (32, 3, 128, 128)


Epoch 08/50 | train loss 0.0269 acc 0.9913 | val loss 0.3129 acc 0.9148 | best_loss 0.2797 | patience 5/5 | time 251.9s
Early stopping triggered at epoch 8
Training done. Best val loss: 0.2796652627375818


In [ ]:
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

model.eval()
all_y, all_pred = [], []

with torch.no_grad():
    for x, y in tqdm(test_loader):
        x = x.to(device)
        logits = model(x)
        pred = logits.argmax(dim=1).cpu().numpy()
        all_pred.extend(pred)
        all_y.extend(y.numpy())

print(classification_report(all_y, all_pred, target_names=class_names))
print("Confusion matrix:\n", confusion_matrix(all_y, all_pred))

100%|██████████| 463/463 [1:36:12<00:00, 12.47s/it]

              precision    recall  f1-score   support

       beige       0.84      0.93      0.88       176
       black       0.97      0.95      0.96      4512
        blue       0.85      0.97      0.91       506
       brown       0.63      0.40      0.49       343
        gray       0.87      0.91      0.89      2272
       green       0.96      0.80      0.87       947
        pink       0.93      0.86      0.89       103
      purple       0.96      0.96      0.96       115
         red       0.92      0.97      0.94      1393
       white       0.94      0.95      0.94      2981
      yellow       0.88      0.91      0.89      1465

    accuracy                           0.92     14813
   macro avg       0.89      0.87      0.88     14813
weighted avg       0.92      0.92      0.92     14813

Confusion matrix:
 [[ 163    0    0    3    5    1    0    0    0    1    3]
 [   0 4304   21   36  129   16    0    0    2    2    2]
 [   0    2  492    0    1    1    0    1    3    1 

In [ ]:
import torch
import os

# misal model kamu
# model = YourModel()

# Tentukan nama file model
model_path = os.path.join(DATA_ROOT, "model_terakhir.pth")

# Simpan
torch.save(model.state_dict(), model_path)

print(f"Model berhasil disimpan di: {model_path}")

Model berhasil disimpan di: /content/drive/MyDrive/MERGED_COLOR_128/model_terakhir.pth
